In [1]:
import pandas as pd

df = pd.read_csv('/Users/alexgarcia/Desktop/CityUniversity/MSDS/DS522 Data Aquisition and Analytics/Modules/Module6/PE06/retail_sales.csv')
df.head()

,TransactionID,Date,CustomerID,Product,Quantity,Price,TotalAmount
0,TID0001,45029,CUST0006,product B,-2,32.88,-72.727928
1,TID0002,45275,NaN,product A,8,32.88,263.040000
2,TID0003,45197,CUST0001,product D,10,32.88,328.800000
3,TID0004,45033,CUST0006,product B,9,32.88,295.920000
4,TID0004,45033,CUST0006,product B,9,32.88,295.920000


In [2]:
# Fill missing CustomerID with 'Unknown'
df['CustomerID'] = df['CustomerID'].fillna('Unknown')

# Drop rows where TotalAmount is missing
df.dropna(subset=['TotalAmount'], inplace=True)

df.head()

,TransactionID,Date,CustomerID,Product,Quantity,Price,TotalAmount
0,TID0001,45029,CUST0006,product B,-2,32.88,-72.727928
1,TID0002,45275,Unknown,product A,8,32.88,263.040000
2,TID0003,45197,CUST0001,product D,10,32.88,328.800000
3,TID0004,45033,CUST0006,product B,9,32.88,295.920000
4,TID0004,45033,CUST0006,product B,9,32.88,295.920000


In [3]:
# Check for duplicates
print("Duplicate rows:", df.duplicated().sum())

# Remove duplicates
df.drop_duplicates(inplace=True)

print("Duplicate rows after removal:", df.duplicated().sum())

Duplicate rows: 1
Duplicate rows after removal: 0


In [4]:
# Correct TotalAmount where it doesn't match Quantity * Price
df['TotalAmount'] = df['Quantity'] * df['Price']

df.head()

,TransactionID,Date,CustomerID,Product,Quantity,Price,TotalAmount
0,TID0001,45029,CUST0006,product B,-2,32.88,-65.76
1,TID0002,45275,Unknown,product A,8,32.88,263.04
2,TID0003,45197,CUST0001,product D,10,32.88,328.80
3,TID0004,45033,CUST0006,product B,9,32.88,295.92
5,TID0005,44998,Unknown,product D,2,32.88,65.76


In [5]:
# Replace negative Quantity with 0 and adjust TotalAmount
df.loc[df['Quantity'] < 0, 'Quantity'] = 0
df['TotalAmount'] = df['Quantity'] * df['Price']

df.head()

,TransactionID,Date,CustomerID,Product,Quantity,Price,TotalAmount
0,TID0001,45029,CUST0006,product B,0,32.88,0.00
1,TID0002,45275,Unknown,product A,8,32.88,263.04
2,TID0003,45197,CUST0001,product D,10,32.88,328.80
3,TID0004,45033,CUST0006,product B,9,32.88,295.92
5,TID0005,44998,Unknown,product D,2,32.88,65.76


In [6]:
# Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Ensure CustomerID is a string
df['CustomerID'] = df['CustomerID'].astype(str)

df.dtypes

TransactionID               str
Date             datetime64[ns]
CustomerID                  str
Product                     str
Quantity                  int64
Price                   float64
TotalAmount             float64
dtype: object

In [7]:
# Extract month from Date
df['Month'] = df['Date'].dt.month

# Categorize transactions by TotalAmount
def categorize_revenue(amount):
    if amount < 50:
        return 'Low'
    elif amount <= 200:
        return 'Medium'
    else:
        return 'High'

df['RevenueCategory'] = df['TotalAmount'].apply(categorize_revenue)

df.head()

,TransactionID,Date,CustomerID,Product,Quantity,Price,TotalAmount,Month,RevenueCategory
0,TID0001,1970-01-01 00:00:00.000045029,CUST0006,product B,0,32.88,0.00,1,Low
1,TID0002,1970-01-01 00:00:00.000045275,Unknown,product A,8,32.88,263.04,1,High
2,TID0003,1970-01-01 00:00:00.000045197,CUST0001,product D,10,32.88,328.80,1,High
3,TID0004,1970-01-01 00:00:00.000045033,CUST0006,product B,9,32.88,295.92,1,High
5,TID0005,1970-01-01 00:00:00.000044998,Unknown,product D,2,32.88,65.76,1,Medium


In [8]:
# Remove rows where Price or Quantity is 0
df = df[(df['Price'] != 0) & (df['Quantity'] != 0)]

df.head()

,TransactionID,Date,CustomerID,Product,Quantity,Price,TotalAmount,Month,RevenueCategory
1,TID0002,1970-01-01 00:00:00.000045275,Unknown,product A,8,32.88,263.04,1,High
2,TID0003,1970-01-01 00:00:00.000045197,CUST0001,product D,10,32.88,328.80,1,High
3,TID0004,1970-01-01 00:00:00.000045033,CUST0006,product B,9,32.88,295.92,1,High
5,TID0005,1970-01-01 00:00:00.000044998,Unknown,product D,2,32.88,65.76,1,Medium
6,TID0006,1970-01-01 00:00:00.000045115,Unknown,product D,8,32.88,263.04,1,High


In [9]:
# Strip whitespace and convert to title case
df['Product'] = df['Product'].str.strip().str.title()

df['Product'].head()

1     Product A
2     Product D
3    Product  B
5     Product D
6     Product D
Name: Product, dtype: str

In [10]:
# Strip whitespace and convert to uppercase
df['CustomerID'] = df['CustomerID'].str.strip().str.upper()

# Replace CustomerIDs starting with 'TEMP' with 'CUST'
df['CustomerID'] = df['CustomerID'].str.replace(r'^TEMP', 'CUST', regex=True)

df['CustomerID'].head()

1     UNKNOWN
2    CUST0001
3    CUST0006
5     UNKNOWN
6     UNKNOWN
Name: CustomerID, dtype: str

In [11]:
# Categorize products based on name content
def categorize_product(name):
    if any(letter in name.upper() for letter in ['A', 'B']):
        return 'Category 1'
    elif any(letter in name.upper() for letter in ['C', 'D']):
        return 'Category 2'
    else:
        return 'Other'

df['ProductCategory'] = df['Product'].apply(categorize_product)

df.head()

,TransactionID,Date,CustomerID,Product,Quantity,Price,TotalAmount,Month,RevenueCategory,ProductCategory
1,TID0002,1970-01-01 00:00:00.000045275,UNKNOWN,Product A,8,32.88,263.04,1,High,Category 1
2,TID0003,1970-01-01 00:00:00.000045197,CUST0001,Product D,10,32.88,328.80,1,High,Category 2
3,TID0004,1970-01-01 00:00:00.000045033,CUST0006,Product B,9,32.88,295.92,1,High,Category 1
5,TID0005,1970-01-01 00:00:00.000044998,UNKNOWN,Product D,2,32.88,65.76,1,Medium,Category 2
6,TID0006,1970-01-01 00:00:00.000045115,UNKNOWN,Product D,8,32.88,263.04,1,High,Category 2


In [12]:
df.to_csv('cleaned_retail_sales.csv', index=False)

print("Cleaned dataset saved to 'cleaned_retail_sales.csv'")
df.head()

Cleaned dataset saved to 'cleaned_retail_sales.csv'


,TransactionID,Date,CustomerID,Product,Quantity,Price,TotalAmount,Month,RevenueCategory,ProductCategory
1,TID0002,1970-01-01 00:00:00.000045275,UNKNOWN,Product A,8,32.88,263.04,1,High,Category 1
2,TID0003,1970-01-01 00:00:00.000045197,CUST0001,Product D,10,32.88,328.80,1,High,Category 2
3,TID0004,1970-01-01 00:00:00.000045033,CUST0006,Product B,9,32.88,295.92,1,High,Category 1
5,TID0005,1970-01-01 00:00:00.000044998,UNKNOWN,Product D,2,32.88,65.76,1,Medium,Category 2
6,TID0006,1970-01-01 00:00:00.000045115,UNKNOWN,Product D,8,32.88,263.04,1,High,Category 2
